In [1]:
# ==========================================
# CELL 1: Environment & Standard Library
# ==========================================
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# PyTorch Ecosystem
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.utils.class_weight import compute_class_weight

# Set device agnostic code (uses GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Computation device set to: {device}")

 Computation device set to: cpu


In [2]:
# set the raw directory folder 
tomato_dir=r"D:\agentic_agriculture\datasets\Tomato_dataset"
print(f"Checking contents of: {tomato_dir}")

# List contents of the base dataset path (e.g., train, validation, test folders)
if os.path.exists(tomato_dir):
    print(f"Contents found at {tomato_dir}:")
    for item in os.listdir(tomato_dir):
        item_path = os.path.join(tomato_dir, item)
        print(f"- {item} {'(Directory)' if os.path.isdir(item_path) else '(File)'}")
else:
    print(f"Base dataset path does not exist: {tomato_dir}")

train = os.path.join(tomato_dir, 'train') # Assuming 'train' is the directory with class folders


print(f"Folders in '{train}':")

# List all entries in the train_dir
for item in os.listdir(train):
    item_path = os.path.join(train, item)
    # Check if the item is a directory (a class folder)
    if os.path.isdir(item_path):
        print(f"- {item}")


Checking contents of: D:\agentic_agriculture\datasets\Tomato_dataset
Contents found at D:\agentic_agriculture\datasets\Tomato_dataset:
- train (Directory)
- valid (Directory)
Folders in 'D:\agentic_agriculture\datasets\Tomato_dataset\train':
- Bacterial_spot
- Early_blight
- healthy
- Late_blight
- Leaf_Mold
- powdery_mildew
- Septoria_leaf_spot
- Spider_mites Two-spotted_spider_mite
- Target_Spot
- Tomato_mosaic_virus
- Tomato_Yellow_Leaf_Curl_Virus


In [3]:



if os.path.exists(train):
    print(f"\nCounting files in each class directory within: {train}")
    class_counts = {}
    for class_name in os.listdir(train):
        class_path = os.path.join(train, class_name)
        if os.path.isdir(class_path):
            # Count only image files (e.g., .jpg, .jpeg, .png)
            num_files = len([name for name in os.listdir(class_path) if name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))])
            class_counts[class_name] = num_files

    # Sort for consistent output
    sorted_class_counts = sorted(class_counts.items())

    for class_name, count in sorted_class_counts:
        print(f"{class_name}: {count} files")
else:
    print(f"The directory '{train}' does not exist. Please check the dataset structure.")


Counting files in each class directory within: D:\agentic_agriculture\datasets\Tomato_dataset\train
Bacterial_spot: 2826 files
Early_blight: 2455 files
Late_blight: 3113 files
Leaf_Mold: 2754 files
Septoria_leaf_spot: 2882 files
Spider_mites Two-spotted_spider_mite: 1747 files
Target_Spot: 1827 files
Tomato_Yellow_Leaf_Curl_Virus: 2039 files
Tomato_mosaic_virus: 2153 files
healthy: 3051 files
powdery_mildew: 1004 files


In [ ]:
#dig into the actual binary and pixel data of those 25,851 images to catch silent 
# killers like corrupted files, exact duplicates, and hidden RGBA/Grayscale formats.

import os
import hashlib
from PIL import Image
from collections import defaultdict

# --- Configuration ---
tomato_dir = r"D:\agentic_agriculture\datasets\Tomato_dataset\train"

# --- Audit Trackers ---
corrupted_files = []
color_modes = defaultdict(int)
image_dimensions = defaultdict(int)
hash_dict = defaultdict(list)
exact_duplicates = []

print(f"Starting Deep Data Audit on: {tomato_dir}\n")

for class_name in os.listdir(tomato_dir):
    class_path = os.path.join(tomato_dir, class_name)
    
    if os.path.isdir(class_path):
        print(f"Auditing class: {class_name}...")
        
        for file_name in os.listdir(class_path):
            if not file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
                continue
                
            file_path = os.path.join(class_path, file_name)
            
            # 1. Exact Duplicate Detection (MD5 Hash)
            try:
                with open(file_path, 'rb') as f:
                    file_bytes = f.read()
                    file_hash = hashlib.md5(file_bytes).hexdigest()
                    hash_dict[file_hash].append(file_path)
            except Exception as e:
                print(f"  [!] Could not read {file_name}: {e}")
                continue
                
            # 2. Image Integrity, Mode, and Dimension Check
            try:
                with Image.open(file_path) as img:
                    img.verify() # Strictly checks for corruption without loading pixel data
                    
                # Re-open to get properties (verify() closes the file)
                with Image.open(file_path) as img:
                    color_modes[img.mode] += 1
                    image_dimensions[img.size] += 1
                    
            except (IOError, SyntaxError) as e:
                corrupted_files.append(file_path)

# --- Process Duplicate Results ---
for file_hash, file_list in hash_dict.items():
    if len(file_list) > 1:
        exact_duplicates.append(file_list)

# --- Print Audit Report ---
print("\n" + "="*40)
print("  DATA AUDIT REPORT")
print("="*40)

print(f"\n Corrupted Images Found: {len(corrupted_files)}")
if corrupted_files:
    print(f"   Sample: {corrupted_files[:3]}...")

print(f"\n Exact Duplicates Found: {len(exact_duplicates)} groups")
if exact_duplicates:
    print(f"   Sample group: {exact_duplicates[0]}")

print("\n Color Modes Distribution:")
for mode, count in color_modes.items():
    print(f"   - {mode}: {count} images")

print("\n Top 5 Image Dimensions:")
sorted_dims = sorted(image_dimensions.items(), key=lambda x: x[1], reverse=True)[:5]
for dim, count in sorted_dims:
    print(f"   - {dim[0]}x{dim[1]}: {count} images")
print("="*40)

Starting Deep Data Audit on: D:\agentic_agriculture\datasets\Tomato_dataset\train

Auditing class: Bacterial_spot...
Auditing class: Early_blight...
Auditing class: healthy...
Auditing class: Late_blight...
Auditing class: Leaf_Mold...
Auditing class: powdery_mildew...
Auditing class: Septoria_leaf_spot...
Auditing class: Spider_mites Two-spotted_spider_mite...
Auditing class: Target_Spot...
Auditing class: Tomato_mosaic_virus...
Auditing class: Tomato_Yellow_Leaf_Curl_Virus...

 📊 DATA AUDIT REPORT

🚨 Corrupted Images Found: 0

👯 Exact Duplicates Found: 622 groups
   Sample group: ['D:\\agentic_agriculture\\datasets\\Tomato_dataset\\train\\Bacterial_spot\\Septoria Leafspot.jpg', 'D:\\agentic_agriculture\\datasets\\Tomato_dataset\\train\\Septoria_leaf_spot\\Septoria Leafspot.jpg']

🎨 Color Modes Distribution:
   - RGB: 25498 images
   - RGBA: 353 images

📏 Top 5 Image Dimensions:
   - 256x256: 18942 images
   - 227x227: 4120 images
   - 640x640: 1207 images
   - 533x800: 151 images
   